## End-to-end tutorial: REINVENT → Maize → GOLD → scores back to REINVENT

This notebook demonstrates the **closed loop** that many practical design workflows need:

1. **REINVENT4** generates a batch of candidate molecules as **SMILES**.
2. **Maize** routes that batch through a graph of nodes.
3. **SMILES → molecules** conversion (so docking tools can consume 3D-ready structures).
4. **GOLD docking** evaluates each molecule against a prepared receptor and binding site.
5. **Score conversion** normalizes docking scores into the numeric range REINVENT expects.
6. The converted scores are **fed back** into REINVENT so the next epoch proposes better molecules.

### What Maize is doing here

Maize acts as the **orchestrator**: each node runs as its own process and communicates via ports/channels. This makes it straightforward to plug a generative model, chemistry conversion, and docking into one repeatable workflow.

### Prerequisites

- Tutorials 1–4 completed.
- A valid **CCDC license**.
- A REINVENT **prior** in `priors/`.
- Prepared protein + reference ligand in `data/` (e.g. `protein_prepared.mol2` and `4lqm_ligand.mol2` from Tutorial 3).
- A `config.toml` in the repo root (copy `config.toml.example` → `config.toml` and fill in tool paths).

### Important


In [ ]:
import workshop_setup
from pathlib import Path
WORKSHOP_ROOT = Path(".").resolve()
WORKSHOP_ROOT

In [ ]:
from maize.core.workflow import Workflow
from pathlib import Path
from maize.steps.mai.molecule import Smiles2Molecules
from maize.steps.mai.misc import ReInvent
from nodes import GOLDDocking, SaveIsomers, ScoreConverter

In [ ]:
### THE PATHS IN THIS NOTEBOOK ARE EXAMPLES. PLEASE CHANGE THEM TO YOUR OWN PATHS ###

flow = Workflow(name='reinvent_dock')
flow.config.update(Path(f'{WORKSHOP_ROOT}/config.toml'))
output_dir = WORKSHOP_ROOT / "data" / "output"
output_dir.mkdir(parents=True, exist_ok=True)
rnve = flow.add(ReInvent)
smi2mol = flow.add(Smiles2Molecules, loop=True)
save = flow.add(SaveIsomers, name='save_mol', parameters={'file': f'{output_dir}/b01a4mols_stage2.sdf'},loop=True)
gold = flow.add(GOLDDocking, loop=True)
converter = flow.add(ScoreConverter, loop=True)

# Set parameters
gold.output_file.set(f'{output_dir}')
gold.protein_file.set(f'{WORKSHOP_ROOT}/data/protein_prepared.mol2')
gold.ref_ligand.set(f'{WORKSHOP_ROOT}/data/4lqm_ligand.mol2')
gold.ndocking.set(20)
gold.scoring_function.set('goldscore')
# Connect the nodes
flow.connect(rnve.out, smi2mol.inp)
flow.connect(smi2mol.out, save.inp)
flow.connect(save.out, gold.inp)
flow.connect(gold.out, converter.inp)
flow.connect(converter.out, rnve.inp) 

# REINVENT configuration
rnve_conf = Path(f'{WORKSHOP_ROOT}/configs/reinvent.toml')
rnve.configuration.set(rnve_conf)
rnve.max_epoch.set(5)  
rnve.low.set(0.0)
rnve.high.set(5)
rnve.reverse.set(False)
rnve.batch_size.set(16) # Number of molecules to generate each epoch




In [ ]:
# Check and visualize the workflow
flow.check()
dot =flow.visualize()
dot


In [ ]:
# Execute the workflow
flow.execute()